# IE 7615 Discriminative Project: live demo (Study Group 6)

**(a)** Given an image of a single object, identify its Object ID (Milestone 1, ConvNeXt-Tiny).
**(b)** Given an image of multiple objects, identify and locate every object (Milestone 2, YOLOv8s).

Run the cells in order. A GPU is optional; CPU is fast enough for single images.

In [ ]:
# Setup: get the code and models (about 1 minute)
!git clone -q https://github.com/sankalphegde/ie7615-discriminative-project.git
%cd ie7615-discriminative-project/milestone3
!pip -q install ultralytics
import sys; sys.path.insert(0, '.')

## (a) Single-object image → Object ID
Run the cell and upload one photo.

In [ ]:
from google.colab import files
import matplotlib.pyplot as plt
from PIL import Image, ImageOps
from identify_object import load_model, identify

clf = load_model()
for name in files.upload():
    preds = identify(clf, name, top=3)
    oid, obj, conf = preds[0]
    plt.figure(figsize=(5, 5)); plt.imshow(ImageOps.exif_transpose(Image.open(name))); plt.axis('off')
    plt.title(f'{oid}  {obj}  ({conf:.1%})', fontsize=14); plt.show()
    for o, n, c in preds[1:]:
        print(f'runner-up: {o}  {n}  ({c:.1%})')

## (b) Multi-object image → ID and location of every object
Run the cell and upload a grid image (or any photo with several objects).

In [ ]:
sys.path.insert(0, '../milestone2')
from pathlib import Path
from ultralytics import YOLO
from detect_objects import detect, load_names

det, names = YOLO('../milestone2/weights/grid_yolov8s_best.pt'), load_names()
for name in files.upload():
    found = detect(det, name, names, conf=0.25, out_dir=Path('shown'))
    plt.figure(figsize=(10, 10)); plt.imshow(Image.open(f'shown/{Path(name).stem}_detected.jpg')); plt.axis('off'); plt.show()
    print(f'{len(found)} object(s):')
    for oid, n, c, (x1, y1, x2, y2) in found:
        print(f'  {oid}  {n:<22s} conf {c:.2f}   box ({x1}, {y1}) to ({x2}, {y2})')